# Benchmark SLM : **SmolLM3-3B** (prompts maison + HumanEval)

Notebook **commun aux 4 membres** (basé sur la version corrigée de Gemma). Seule différence : `MODEL_KEY = "smollm"`.

- **Partie A** : 15 prompts maison -> `results/benchmark_smollm3-3b.csv`
- **Partie B** : 20 problèmes HumanEval -> `results/humaneval_smollm3-3b.json`
- **Partie C** : notation manuelle (bugs, qualité, hallucinations)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # un seul GPU (Kaggle en donne 2) -> mesures comparables

REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "correctifs-gemma"   # branche contenant les derniers correctifs communs

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)   # Git ne garde pas les dossiers vides

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 53 (delta 14), reused 26 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 35.16 KiB | 5.02 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
!pip install -r ../requirements.txt -q
!pip install -q "transformers>=4.53"   # SmolLM3 n'est supporte qu'a partir de transformers 4.53
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-b707dc3f-c4c4-6441-be5b-e89b39b61340)
GPU 1: Tesla T4 (UUID: GPU-deb85702-10a8-7af9-b3b7-eef4842bf1de)


## SmolLM3 : pas de login Hugging Face
Contrairement à Gemma, **SmolLM3-3B est un modèle public** : pas de token à créer, pas de cellule `login`.

In [3]:
import json, time, sys, re
import torch
import pandas as pd
import transformers
from packaging import version as _v
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

assert _v.parse(transformers.__version__) >= _v.parse("4.53.0"), \
    f"transformers {transformers.__version__} trop ancien pour SmolLM3 : relancez la cellule pip puis redemarrez la session."

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "smollm"
# ====================================================================

with open("../config.json") as f:
    config = json.load(f)

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)   # "/no_think" pour SmolLM3 seulement
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME)
print("System prompt :", SYSTEM_PROMPT, "| transformers", transformers.__version__)

HuggingFaceTB/SmolLM3-3B | smollm3-3b | Tesla T4
System prompt : /no_think | transformers 5.16.1


In [5]:
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
DTYPE = torch.bfloat16 if config["precision"] == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=DTYPE
)
print("Précision :", config["precision"])
model.eval()


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Échauffement + contrôle : le 1er appel est lent, et on vérifie que le modèle répond vraiment
test = generate("Write a Python function add(a, b) that returns a + b.")
print(test["response"][:300])
assert "def" in test["response"], "Réponse vide ou incohérente : verifiez la version de transformers et redemarrez la session."
if "<think>" in test["response"]:
    print("ATTENTION : balises <think> dans la reponse -> le mode /no_think ne semble pas actif.")
print("Modele pret.")

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Précision : bf16


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Here is a Python function that adds two numbers:

```python
def add(a, b):
    """
    Returns the sum of two numbers.

    Args:
        a (int or float): The first number.
        b (int or float): The second number.

    Returns:
        int or float: The sum of a and b.
    """
    return a + b

Modele pret.


## Partie A : 15 prompts maison

In [6]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]

def norm(s):  # pour comparer sans tenir compte des espaces
    return re.sub(r"\s+", "", s)

results = []
for p in prompts:
    gen = generate(p["prompt"])
    response = gen.pop("response")

    status, error = None, None
    # Cas 1 : generation de code 
    if p["task"] == "generation_code" and "test" in p:
        status, error = run_tests(extract_code(response), p["test"],
                                  timeout=HE_CONFIG["timeout_seconds"])
        # Cas 2 : completion_code
    elif p["task"] == "completion_code" and "expected" in p:
        status = "pass" if norm(p["expected"]) in norm(response) else "fail"
    # bug_detection, explication, refactoring, generation_tests : notes manuelles

    results.append({
        "id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": response,
        **gen,
        "auto_status": status, "auto_error": error,
        # A remplir a la main (voir grille_notation.md et README)
        "bug_label": None, "note_qualite": None, "hallucination": None,
    })
    print(p["id"], status)

df_prompts = pd.DataFrame(results)
df_prompts.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
print(f"-> results/benchmark_{model_slug}.csv")
df_prompts[["id", "task", "ttft_s", "latency_s", "tokens_per_sec", "gpu_memory_go", "auto_status"]]

P1 pass
P2 pass
phrase une est Ceci
P3 pass
6
P4 pass
P5 pass
P6 pass
P7 None
P8 None
P9 None
P10 None
P11 None
P12 None
P13 None
P14 None
P15 None
-> results/benchmark_smollm3-3b.csv


,id,task,ttft_s,latency_s,tokens_per_sec,gpu_memory_go,auto_status
0,P1,generation_code,0.417,11.250,21.42,6.19,pass
1,P2,generation_code,0.308,11.487,21.50,6.20,pass
2,P3,generation_code,0.308,7.605,21.04,6.19,pass
3,P4,generation_code,0.316,10.586,21.25,6.19,pass
4,P5,completion_code,0.321,4.382,20.08,6.18,pass
5,P6,completion_code,0.327,6.666,20.70,6.19,pass
6,P7,bug_detection,0.316,8.768,20.64,6.19,None
7,P8,bug_detection,0.321,10.635,21.16,6.19,None
8,P9,bug_detection,0.309,4.030,20.35,6.18,None
9,P10,explication,0.318,3.320,19.88,6.18,None


## Partie B : HumanEval (20 problèmes communs)

In [7]:
with open("../humaneval_subset.json") as f:
    subset = json.load(f)

humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
print(len(humaneval), "problemes")

HEADER = "from typing import *\nimport math\n\n"   # imports courants que le modele peut oublier

details = []
for ex in humaneval:
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                              timeout=HE_CONFIG["timeout_seconds"])
    details.append({
        "task_id": ex["task_id"], "status": status, "error": error,
        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
        "tokens_per_sec": gen["tokens_per_sec"],
        "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"],
        "response": gen["response"],
    })
    print(ex["task_id"], status)

n_pass = sum(d["status"] == "pass" for d in details)
summary = {
    "model": MODEL_NAME,
    "gpu": GPU_NAME,
    "precision": config["precision"],
    "dataset": "HumanEval (sous-ensemble des 20 premiers problemes)",
    "n_problems": len(details),
    "n_pass": n_pass,
    "pass@1": round(100 * n_pass / len(details), 1),
    "details": details,
}
with open(f"../results/humaneval_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"pass@1 = {summary['pass@1']}% ({n_pass}/{len(details)}) -> results/humaneval_{model_slug}.json")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

20 problemes
HumanEval/0 pass
HumanEval/1 fail
HumanEval/2 pass
HumanEval/3 pass
HumanEval/4 pass
HumanEval/5 pass
HumanEval/6 pass
HumanEval/7 pass
HumanEval/8 pass
HumanEval/9 pass
HumanEval/10 fail
HumanEval/11 pass
HumanEval/12 pass
HumanEval/13 pass
HumanEval/14 fail
HumanEval/15 pass
HumanEval/16 pass
HumanEval/17 pass
HumanEval/18 pass
HumanEval/19 pass
pass@1 = 85.0% (17/20) -> results/humaneval_smollm3-3b.json


In [8]:
for d in details:
    if d["status"] != "pass":
        print("=" * 80)
        print(d["task_id"], "|", d["status"], "|", d["error"])
        print("--- REPONSE DU MODELE :")
        print(d["response"])

HumanEval/1 | fail | AssertionError: 
--- REPONSE DU MODELE :
```python
from typing import List

def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open brace is properly closed) and not nested within each other
    Ignore any spaces in the input string.
    >>> separate_paren_groups('( ) (( )) (( )( ))')
    ['()', '(())', '(()())']
    """
    # Remove spaces from the input string
    paren_string = paren_string.replace(' ', '')
    
    # Initialize an empty list to store the groups
    groups = []
    
    # Initialize a stack to keep track of the current group
    stack = []
    
    # Initialize a variable to keep track of the current group
    current_group = []
    
    # Iterate over each character in the string
    for char in paren_string:
      

Les 3 échecs sont des erreurs de raisonnement du modèle (aucun timeout, aucune réponse tronquée, aucun problème d’extraction).

In [9]:
# Fichier d'environnement PROPRE A CHAQUE MEMBRE (pas d'ecrasement entre vous)
from importlib.metadata import version
import platform
lines = [f"# Environnement - {MODEL_NAME}", "",
         f"- GPU : {GPU_NAME}",
         f"- Python : {platform.python_version()}",
         f"- torch : {torch.__version__}",
         f"- CUDA : {torch.version.cuda}"]
for lib in ["transformers", "accelerate", "huggingface_hub", "datasets", "pandas"]:
    lines.append(f"- {lib} : {version(lib)}")
open(f"../environnement_{model_slug}.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

# Environnement - HuggingFaceTB/SmolLM3-3B

- GPU : Tesla T4
- Python : 3.13.15
- torch : 2.11.0+cu128
- CUDA : 12.8
- transformers : 5.16.1
- accelerate : 1.14.0
- huggingface_hub : 1.29.0
- datasets : 4.8.5
- pandas : 2.3.3


## Partie C : notation manuelle (voir `grille_notation.md`)
1. Lisez la cellule d'affichage ci-dessous (prompt + réponse de chaque prompt).
2. Remplissez les 3 dictionnaires de la cellule suivante, puis exécutez-la : elle écrit les notes dans le CSV.

In [10]:
df = pd.read_csv(f"../results/benchmark_{model_slug}.csv")
for _, r in df.iterrows():
    print("=" * 90)
    err = r["auto_error"] if pd.notna(r["auto_error"]) else ""
    print(r["id"], "|", r["task"], "| auto :", r["auto_status"], err)
    print("--- PROMPT :"); print(r["prompt"])
    print("--- REPONSE :"); print(r["response"])

P1 | generation_code | auto : pass 
--- PROMPT :
Écris une fonction Python is_prime(n) qui retourne True si n est premier, False sinon.
--- REPONSE :
Voici une fonction Python `is_prime(n)` qui vérifie si un nombre `n` est premier ou non :

```python
def is_prime(n):
    """
    Vérifie si un nombre n est premier.
    
    :param n: Un entier
    :return: True si n est premier, False sinon
    """
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True
```

Cette fonction utilise une méthode optimisée pour vérifier si un nombre est premier. Elle commence par vérifier les cas particuliers (n <= 1, n <= 3, etc.) et ensuite utilise une boucle pour tester la divisibilité par des nombres de la forme 6k ± 1, ce qui réduit le nombre de tests nécessaires.
P2 | generation_code | auto : pass 
--- PR

- "non" : il y a du code, et tout ce qu’il utilise existe vraiment.

- "oui" : il y a du code, mais il contient quelque chose d’inventé (par exemple liste.trier_vite() ou import super_lib).

- None : il n’y a pas de code écrit par le modèle, donc la question ne s’applique pas.

In [13]:
# P7-P9 : "TP", "FP", "FN" ou "TN"
BUG_LABELS = {"P7": "TP", "P8": "TP", "P9": "TN"}
# P10-P15 : note de 1 a 5 (P14-P15 = tests generes)
NOTES = {"P10": 4, "P11": 5, "P12": 5, "P13": 3, "P14": 4, "P15": 3}

HALLU = {
    "P1": "non", "P2": "non", "P3": "non", "P4": "non", "P5": "non", "P6": "non",
    "P7": "non", "P8": "non", "P9": None, "P10": None, "P11": None,
    "P12": "non", "P13": "non", "P14": "non", "P15": "non",
}
# -----------------------------------------------------------

for c in ["bug_label", "note_qualite", "hallucination"]:
    df[c] = df[c].astype(object)

def apply(col, mapping, allowed):
    for pid, val in mapping.items():
        if val is None:
            continue
        assert val in allowed, f"{pid} : valeur invalide {val!r} (attendu : {allowed})"
        assert pid in set(df["id"]), f"{pid} inconnu"
        df.loc[df["id"] == pid, col] = val

apply("bug_label", BUG_LABELS, {"TP", "FP", "FN", "TN"})
apply("note_qualite", NOTES, {1, 2, 3, 4, 5})
apply("hallucination", HALLU, {"oui", "non"})

df.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
manquants = [(c, int(df[c].isna().sum())) for c in ["bug_label", "note_qualite", "hallucination"]]
print("Valeurs encore vides (colonne, nb) :", manquants)

Valeurs encore vides (colonne, nb) : [('bug_label', 12), ('note_qualite', 9), ('hallucination', 3)]


Le CSV est un tableau de **15 lignes** (P1 à P15). Les 3 colonnes de notes étaient vides au départ. Après le remplissage, le programme compte **combien de cases restent vides** dans chaque colonne.

| Colonne | Cases vides | Pourquoi |
|----------|------------|-----------|
| `bug_label` | 12 | on a rempli 3 lignes (P7, P8, P9), donc 15 − 3 = 12 vides. Les autres prompts ne concernent pas la détection de bugs. |
| `note_qualite` | 9 | on a rempli 6 lignes (P10 à P15), donc 15 − 6 = 9 vides. |
| `hallucination` | 3 | on a rempli 12 lignes, donc il reste P9, P10 et P11 sans valeur renseignée. |

## Récupérer vos fichiers
Le zip contient **uniquement vos 3 fichiers** (à ajouter au dépôt, dans `results/` et à la racine de `benchmarking/`).

In [14]:
pd.read_csv(f"../results/benchmark_{model_slug}.csv")[["id","bug_label","note_qualite","hallucination"]]

,id,bug_label,note_qualite,hallucination
0,P1,NaN,NaN,non
1,P2,NaN,NaN,non
2,P3,NaN,NaN,non
3,P4,NaN,NaN,non
4,P5,NaN,NaN,non
5,P6,NaN,NaN,non
6,P7,TP,NaN,non
7,P8,TP,NaN,non
8,P9,TN,NaN,NaN
9,P10,NaN,4.0,NaN


In [15]:
import zipfile
files = {f"../results/benchmark_{model_slug}.csv": f"results/benchmark_{model_slug}.csv",
         f"../results/humaneval_{model_slug}.json": f"results/humaneval_{model_slug}.json",
         f"../environnement_{model_slug}.md": f"environnement_{model_slug}.md"}
out = f"/kaggle/working/resultats_{model_slug}.zip" if os.path.exists("/kaggle/working") else f"resultats_{model_slug}.zip"
with zipfile.ZipFile(out, "w") as z:
    for src, arc in files.items():
        z.write(src, arcname=arc)
print("Cree :", out)

Cree : /kaggle/working/resultats_smollm3-3b.zip
